# T15 — Wall-clock timing benchmark

**Question answered.** Sections 4.6 and 4.8 of the manuscript claim "milliseconds,"
"seconds on an ordinary laptop," and compare against alignment-based tools' O(n x m) cost --
none of it measured. This notebook times every locally-runnable model end to end: training
time, per-sequence inference time (batch of 1,000 test sequences), and peak RAM, on both CPU
and GPU where relevant. It reuses the Kameris-equivalent and CASTOR-equivalent
re-implementations from the T5 notebook so the timing table lines up with the accuracy table
there.

**What this notebook can and cannot time.** Linear SVM, Random Forest, and the three CNNs
(CPU and GPU) are timed directly here. COMET, REGA, and REGA are external web tools with no
bulk API (see T5) -- their timing cannot be automated from inside Kaggle. If you run them
manually via T5's FASTA export, note the wall-clock time the web tool itself reports (COMET
in particular reports its own runtimes) and add those rows to the table in Section 7 by hand;
this notebook leaves clearly marked placeholder rows for that.

**GPU needed.** Only for the GPU-side CNN timing rows. The notebook will still produce the
full CPU-only timing table without a GPU; if you want the CPU-vs-GPU CNN comparison (the
number that matters most for the deployability argument, since target settings are
resource-limited), turn on GPU T4 x2.

**Runtime.** The timing runs themselves are fast (they ARE the thing being measured — a few
minutes total for classical models; CNN train+infer timing adds the CNNs' usual few minutes
each, once per hardware mode).

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, re, time, platform, resource
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

print("CPU:", platform.processor() or platform.machine())
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h | head -2
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "No GPU visible."


### 2. Load data, fixed split (same as everywhere else)

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
n_cls = len(le.classes_)
X_all = df["sequence"].values
grp_all = df["cluster"].values

s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
trv, te = next(s1.split(X_all, y_all, grp_all))
ytr, yte = y_all[trv], y_all[te]

N_TIMING = 1000
timing_idx = te[:N_TIMING] if len(te) >= N_TIMING else te
print(f"Timing inference on {len(timing_idx)} test sequences.")


### 3. Peak-RAM helper + median-of-5 timing helper

In [ ]:

def peak_rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # KB -> MB on Linux

def time_median(fn, n_runs=5):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


### 4. Time the classical models (SVM, Random Forest) — CPU

In [ ]:

vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=512, lowercase=False)
Xtr = vec.fit_transform(X_all[trv])
Xte_timing = vec.transform(X_all[timing_idx])

rows = []
for name, clf_builder in [
    ("linear_svm", lambda: LinearSVC(class_weight="balanced", random_state=0, max_iter=20000)),
    ("random_forest", lambda: RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                                        random_state=0, n_jobs=-1)),
]:
    rss0 = peak_rss_mb()
    clf = clf_builder()
    train_t = time_median(lambda: clf.fit(Xtr, ytr), n_runs=1)  # 1 run, training is deterministic+slow
    clf.fit(Xtr, ytr)  # ensure fitted for the inference timing below
    infer_t_total = time_median(lambda: clf.predict(Xte_timing), n_runs=5)
    rss1 = peak_rss_mb()
    rows.append(dict(method=name, hardware="CPU", train_seconds=train_t,
                      inference_seconds_per_1000seq=infer_t_total,
                      inference_ms_per_seq=1000*infer_t_total/len(timing_idx),
                      peak_ram_mb=rss1, gpu_required="no"))
    print(rows[-1])


### 5. Time the CNNs — CPU and, if available, GPU

Reuses the same lightweight 1D-CNN builders as the T1 notebook.

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, Model, Input

def conv_bn_relu(x, filters, kernel=5, stride=1):
    x = layers.Conv1D(filters, kernel, strides=stride, padding="same")(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

def build_resnet18_1d(input_len, n_classes):
    inp = Input(shape=(input_len, 1))
    x = conv_bn_relu(inp, 32, 7, 2)
    x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
    for filters, stride in [(32,1),(32,1),(64,2),(64,1),(128,2),(128,1),(256,2),(256,1)]:
        shortcut = x
        y = conv_bn_relu(x, filters, 3, stride)
        y = layers.Conv1D(filters, 3, padding="same")(y); y = layers.BatchNormalization()(y)
        if shortcut.shape[-1] != filters or stride != 1:
            shortcut = layers.Conv1D(filters, 1, strides=stride, padding="same")(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        x = layers.Activation("relu")(layers.Add()([shortcut, y]))
    x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return Model(inp, out)

def time_cnn_on_device(device_str, hardware_label):
    with tf.device(device_str):
        Xtr_d = np.asarray(Xtr.todense(), dtype=np.float32)[..., None]
        Xte_d = np.asarray(Xte_timing.todense(), dtype=np.float32)[..., None]
        model = build_resnet18_1d(Xtr_d.shape[1], n_cls)
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
        train_t = time_median(lambda: model.fit(Xtr_d, ytr, epochs=1, batch_size=48, verbose=0), n_runs=1)
        _ = model.predict(Xte_d, verbose=0)  # warm up (graph tracing shouldn't count against timing)
        infer_t = time_median(lambda: model.predict(Xte_d, verbose=0), n_runs=5)
        return dict(method="resnet18_1d", hardware=hardware_label, train_seconds_per_epoch=train_t,
                    inference_seconds_per_1000seq=infer_t,
                    inference_ms_per_seq=1000*infer_t/len(timing_idx),
                    gpu_required=("yes" if hardware_label=="GPU" else "no"))

cpu_row = time_cnn_on_device("/CPU:0", "CPU")
rows.append(cpu_row); print(cpu_row)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    gpu_row = time_cnn_on_device("/GPU:0", "GPU")
    rows.append(gpu_row); print(gpu_row)
else:
    print("No GPU visible in this session -- skipping GPU timing row. "
          "Turn on GPU T4 x2 in notebook settings and re-run this cell to fill it in.")


### 6. Placeholder rows for COMET / REGA / Kameris / CASTOR (fill in from T5's manual step, or from the tools' own reported runtimes)

In [ ]:

external_placeholders = [
    dict(method="comet", hardware="web_tool", train_seconds=np.nan,
         inference_seconds_per_1000seq=np.nan, inference_ms_per_seq=np.nan, gpu_required="no",
         note="Fill in from COMET's own reported runtime (COMET prints per-run timing) "
              "after the manual T5 submission step."),
    dict(method="rega", hardware="web_tool", train_seconds=np.nan,
         inference_seconds_per_1000seq=np.nan, inference_ms_per_seq=np.nan, gpu_required="no",
         note="REGA does not report per-sequence timing on its web interface; time the "
              "wall-clock from submission to result download manually and record it here."),
    dict(method="kameris_equivalent", hardware="CPU", train_seconds=np.nan,
         inference_seconds_per_1000seq=np.nan, inference_ms_per_seq=np.nan, gpu_required="no",
         note="Copy the timing rows already measured for this in the T5 notebook "
              "(kameris_rows), which time the identical local re-implementation."),
    dict(method="castor_equivalent", hardware="CPU", train_seconds=np.nan,
         inference_seconds_per_1000seq=np.nan, inference_ms_per_seq=np.nan, gpu_required="no",
         note="Copy the timing rows already measured for this in the T5 notebook (castor_rows)."),
]
for r in external_placeholders:
    rows.append(r)


### 7. Final table

In [ ]:

timing_table = pd.DataFrame(rows)
timing_table.to_csv(f"{OUT}/t15_timing_benchmark.csv", index=False)
timing_table


### 8. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r t15_timing_outputs.zip t15_timing_benchmark.csv